# Notebook 29. Clearer Yamazaki-Style Blocking Process

This notebook turns the Yamazaki-style East Siberian blocking story into a cleaner, more sequential case-study workflow for the `1-10 February 2018` evolution.

Instead of packing too many ideas into one overloaded panel, Notebook 29 separates the blocking process into the pieces we actually want to explain:

- `500 hPa` height-anomaly shading with `500 hPa` height contours
- actual `300 hPa` wind vectors for the upper-level circulation
- a separate cold-air mass-flux panel below the `280 K` threshold
- `SLP` anomaly shading with explicit `H` / `L` centers
- `850 hPa` temperature-anomaly contours with actual low-level winds
- a marked East Siberian blocking box, a simple ridge-axis annotation, and a summary panel that tells the story in order

The goal here is explanation first. We are not rebuilding the full blocking index or regression framework unless it becomes necessary later. If the final synoptic story matches the paper's logic, we have a clearer and easier-to-defend reproduction.
            

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = os.environ.get("JPCZ_CATALOG_BRANCH", "codex/notebook16-pcolormesh")
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")


def clone_repo_branch():
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )


def sync_repo_branch():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)


if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    clone_repo_branch()
else:
    print("Using existing repo clone:", REPO_DIR)

try:
    sync_repo_branch()
except subprocess.CalledProcessError:
    print("Existing clone could not switch branches cleanly. Re-cloning target branch.")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    clone_repo_branch()
    sync_repo_branch()

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

active_branch = subprocess.run(["git", "-C", REPO_DIR, "branch", "--show-current"], text=True, capture_output=True, check=True).stdout.strip()
active_commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True, capture_output=True, check=True).stdout.strip()
print("Active branch:", active_branch)
print("Active commit:", active_commit)
            

In [ ]:
from pathlib import Path
import importlib
import shutil

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from jpcz_catalog.era5 import open_arco_era5
import jpcz_catalog.blocking as blocking_module
blocking_module = importlib.reload(blocking_module)

BLOCKING_EXPORT_DIR = Path("outputs/verification/blocking_process_story")
BLOCKING_FIGURE_DIR = BLOCKING_EXPORT_DIR / "figures"
BLOCKING_CLIMATOLOGY_DIR = BLOCKING_EXPORT_DIR / "climatology"
BLOCKING_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
BLOCKING_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
BLOCKING_CLIMATOLOGY_DIR.mkdir(parents=True, exist_ok=True)


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def save_figure(fig, path: Path, *, dpi: int = 170):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    maybe_copy_to_drive(path, verbose=False)
    print("Saved figure:", path)
            

## Case Settings

These defaults treat Notebook 29 as a February 2018 blocking-process case study first. You can widen the dates or change the time step later if we want to turn it into a broader blocking notebook.
            

In [ ]:
CASE_START_UTC = "2018-02-01T00:00:00"
CASE_END_UTC = "2018-02-10T00:00:00"
MEAN_START_UTC = "2018-02-01T00:00:00"
MEAN_END_UTC = "2018-02-05T00:00:00"
EVOLUTION_STEP_HOURS = 24
THETA_THRESHOLD_K = 280.0
CLIMATOLOGY_YEARS = tuple(range(1980, 2019))
CLIMATOLOGY_MONTHS = (2,)
ERA5_TIME_CHUNK = 24
SAVE_FIGURES = False
DETAILED_TIME_INDEX = 4

print("Case window:", CASE_START_UTC, "to", CASE_END_UTC)
print("Mean-figure window:", MEAN_START_UTC, "to", MEAN_END_UTC)
print("Evolution step [h]:", EVOLUTION_STEP_HOURS)
print("Theta threshold [K]:", THETA_THRESHOLD_K)
print("Climatology years:", f"{min(CLIMATOLOGY_YEARS)}-{max(CLIMATOLOGY_YEARS)} ({len(CLIMATOLOGY_YEARS)} years)")
            

In [ ]:
for file_name in blocking_module.BLOCKING_CLIMATOLOGY_FILE_NAMES.values():
    path = BLOCKING_CLIMATOLOGY_DIR / file_name
    if not path.exists():
        restore_from_drive_cache(path)

era5_runtime_ds = None


def get_era5_runtime_ds():
    global era5_runtime_ds
    if era5_runtime_ds is None:
        era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
    return era5_runtime_ds


climatology_bundle, era5_runtime_ds = blocking_module.load_or_update_blocking_climatology_bundle(
    BLOCKING_CLIMATOLOGY_DIR,
    years=CLIMATOLOGY_YEARS,
    months=CLIMATOLOGY_MONTHS,
    current_ds=get_era5_runtime_ds(),
    chunks={"time": ERA5_TIME_CHUNK},
)

for file_name in blocking_module.BLOCKING_CLIMATOLOGY_FILE_NAMES.values():
    maybe_copy_to_drive(BLOCKING_CLIMATOLOGY_DIR / file_name, verbose=False)

print("Blocking climatologies ready:")
for variable_name, file_name in blocking_module.BLOCKING_CLIMATOLOGY_FILE_NAMES.items():
    print("-", variable_name, BLOCKING_CLIMATOLOGY_DIR / file_name)
            

## Build The February 2018 Diagnostic Bundles

This cell creates one blocking-diagnostic bundle per selected analysis time, then averages the `1-5 February 2018` subset into one mean figure that is easier to compare against Yamazaki's blocking-process logic.
            

In [ ]:
case_times = pd.date_range(CASE_START_UTC, CASE_END_UTC, freq=f"{EVOLUTION_STEP_HOURS}H")
snapshot_bundles = [
    blocking_module.build_blocking_snapshot_bundle(
        get_era5_runtime_ds(),
        analysis_time,
        climatology_bundle,
        theta_threshold_k=THETA_THRESHOLD_K,
    )
    for analysis_time in case_times
]

mean_time_lookup = {
    pd.Timestamp(time_value)
    for time_value in pd.date_range(MEAN_START_UTC, MEAN_END_UTC, freq=f"{EVOLUTION_STEP_HOURS}H")
}
mean_source_bundles = [
    bundle for bundle in snapshot_bundles if pd.Timestamp(bundle["analysis_time"]) in mean_time_lookup
]
if not mean_source_bundles:
    raise RuntimeError("The requested mean-figure window did not overlap the case-study times.")

mean_bundle = blocking_module.average_story_bundles(
    mean_source_bundles,
    analysis_label=f"{pd.Timestamp(MEAN_START_UTC):%Y-%m-%d} to {pd.Timestamp(MEAN_END_UTC):%Y-%m-%d} mean",
)

daily_summary_df = pd.DataFrame(
    [blocking_module.build_blocking_compact_row(bundle) for bundle in snapshot_bundles]
)

print("Built", len(snapshot_bundles), "snapshot bundles")
print("Daily times:")
for bundle in snapshot_bundles:
    print("-", bundle["analysis_label"])
            

## Mean Blocking Story

This is the main clearer-Yamazaki figure: upper ridge, upper-level circulation, cold-air flux, lower-level pressure response, lower-level thermal response, and a short sequential interpretation.
            

In [ ]:
mean_summary_df = blocking_module.build_blocking_summary_df(mean_bundle)
display(mean_summary_df)

display(Markdown("### Sequential interpretation"))
for line in blocking_module.build_blocking_story_lines(mean_bundle):
    display(Markdown(f"- {line}"))

mean_fig = blocking_module.plot_blocking_story_figure(mean_bundle)
plt.show()

if SAVE_FIGURES:
    save_figure(
        mean_fig,
        BLOCKING_FIGURE_DIR / f"blocking_story_mean_{pd.Timestamp(MEAN_START_UTC):%Y%m%d}_{pd.Timestamp(MEAN_END_UTC):%Y%m%d}.png",
    )
            

## Evolution Gallery A

This gallery keeps the story compact: left is the `500 hPa` blocking-ridge panel and right is the lower-level `SLP` response with the same larger-domain framing.
            

In [ ]:
upper_surface_fig = blocking_module.plot_upper_surface_evolution_gallery(snapshot_bundles)
plt.show()

if SAVE_FIGURES:
    save_figure(upper_surface_fig, BLOCKING_FIGURE_DIR / "blocking_story_upper_surface_evolution.png")
            

## Evolution Gallery B

This second gallery keeps the jet-circulation panel and the cold-air mass-flux panel separate, so the vector story is easier to follow than in the original figure.
            

In [ ]:
jet_flux_fig = blocking_module.plot_jet_flux_evolution_gallery(snapshot_bundles)
plt.show()

if SAVE_FIGURES:
    save_figure(jet_flux_fig, BLOCKING_FIGURE_DIR / "blocking_story_jet_flux_evolution.png")
            

## Daily Metric Table

This is the compact time-by-time table for the same case-study window. It is useful when we want to compare the visual interpretation to a simple set of blocking cues without invoking the full published index.
            

In [ ]:
display(daily_summary_df)
            

## Selected-Time Deep Dive

Use `DETAILED_TIME_INDEX` from the settings cell to pull one of the daily snapshots back into the full six-panel figure.
            

In [ ]:
if not 0 <= int(DETAILED_TIME_INDEX) < len(snapshot_bundles):
    raise IndexError(f"DETAILED_TIME_INDEX must be between 0 and {len(snapshot_bundles) - 1}.")

selected_bundle = snapshot_bundles[int(DETAILED_TIME_INDEX)]
display(Markdown(f"### Selected time: {selected_bundle['analysis_label']}"))
display(blocking_module.build_blocking_summary_df(selected_bundle))
for line in blocking_module.build_blocking_story_lines(selected_bundle):
    display(Markdown(f"- {line}"))

selected_fig = blocking_module.plot_blocking_story_figure(selected_bundle)
plt.show()

if SAVE_FIGURES:
    save_figure(
        selected_fig,
        BLOCKING_FIGURE_DIR / f"blocking_story_selected_{pd.Timestamp(selected_bundle['analysis_time']):%Y%m%d_%H%M}.png",
    )
            